# 01 · Datasets and fixed-point HAR
Public accelerometer and gyroscope recordings drive the entire experiment. HAPT supplies waist-mounted smartphone IMU recordings. MotionSense supplies pocket-phone Core Motion data. The experiment uses six activity classes per dataset. HAPT transition annotations are excluded from the six-class benchmark.

The target participant contributes only an early trial to calibration. Test windows come from separate later trials. The source feature normalization and prior class prototypes use other participants only. The three seeds change calibration excerpts and simulated annotation mistakes.

In [1]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "har").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"

from har.data import DATASETS, FILES, LABELS, prepare, segments, windows_of
from har.experiments import SEEDS, calibration_indices
from har.model import DIM, VECTORS, fit_metric
from har.native import c_embeddings, features

print("Seeds:", SEEDS)
display(pd.read_csv(RESULTS / "tables" / "dataset_inventory.csv"))

Seeds: (17, 43, 101)


,Dataset,Participants,Test windows,Calibration windows/person
0,HAPT,30,12604,96
1,MotionSense,24,12524,96


## Recompute the input pipeline
The loader verifies both archives against pinned SHA-256 hashes. The C encoder computes 24 fixed-point features from 100 samples. Windows advance by 25 samples and form four-vector editing blocks.

In [2]:
data = {name: prepare(name) for name in DATASETS}
inventory = []
for name, values in data.items():
    candidates = np.unique(values["subject"])
    for subject in candidates:
        calibration = calibration_indices(values, int(subject), np.random.default_rng([17, subject]))
        assert not values["test"][calibration].any()
        assert set(values["trial"][calibration]).isdisjoint(
            values["trial"][(values["subject"] == subject) & values["test"]]
        )
    inventory.append({
        "Dataset": name,
        "Subjects": len(candidates),
        "Raw feature windows": values["raw"].shape[0] * VECTORS,
        "Archive SHA-256": FILES[name][2],
    })
display(pd.DataFrame(inventory))

,Dataset,Subjects,Raw feature windows,Archive SHA-256
0,HAPT,30,25648,4ac4ae064227c07045a99551876b54d204837e995e298b...
1,MotionSense,24,54720,bc632abddbb1e92db31daa7b740710bc8f756f90febe38...


In [3]:
rows = []
for name, values in data.items():
    source = values["subject"] != 1
    metric = fit_metric(values["raw"][source].reshape(-1, DIM))
    segment = next(s for s in segments(name) if s.subject == 1 and s.test and len(s.samples) >= 175)
    windows = windows_of(segment.samples)[:4]
    encoded = features(windows)
    np.testing.assert_array_equal(metric.apply(encoded), c_embeddings(encoded, metric))
    for i, activity in enumerate(LABELS[name]):
        rows.append({"Dataset": name, "Activity": activity,
                     "Test windows": int(np.sum(values["test"] & (values["label"] == i))) * VECTORS})
display(pd.DataFrame(rows).pivot(index="Activity", columns="Dataset", values="Test windows").fillna(0).astype(int))
print("The Python and C integer embeddings agree exactly on both raw-data fixtures.")

The Python and C integer embeddings agree exactly on both raw-data fixtures.


Dataset,HAPT,MotionSense
Activity,,
Jog,0,1068
Lie,2436,0
Sit,2152,4048
Stairs down,1684,780
Stairs up,1816,1000
Stand,2448,3140
Walk,2068,2488


Dataset archives and their terms are documented in [DATASETS.md](../DATASETS.md). Recomputing every participant–seed run takes a single command from the repository root.

```bash
python -m har.experiments
```